# Chapter 4: Training a Neural Network and Computational Thinking

In [1]:
import torch
import matplotlib.pyplot as plt
import numpy as np
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
# At the top of your notebook, add:
np.set_printoptions(suppress=True, precision=8)
torch.set_printoptions(sci_mode=False, precision=8)

cuda


# 1. Understanding Modules and Layers in PyTorch

## Parameter Registration

Correct way to do parameter registration is commented in the following code-snippet and it will run with error.

In [5]:
class BrokenLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        # Just a regular tensor
        self.weight = torch.randn(out_dim, in_dim)
        self.bias = torch.zeros(out_dim)
        
        # Correct way.
        # self.weight = nn.Parameter(torch.randn(out_dim, in_dim))
        # self.bias = nn.Parameter(torch.zeros(out_dim))
        
    def forward(self, x):
        return x @ self.weight.T + self.bias

layer = BrokenLayer(10, 5)
print(f"Number of parameters: {sum(p.numel() for p in layer.parameters())}")  # 0
print(f"Requires grad on weight? {layer.weight.requires_grad}")  # False

# The optimizer will have nothing to optimize!
optimizer = torch.optim.SGD(layer.parameters(), lr=0.01)
print(f"Optimizer param groups: {len(optimizer.param_groups[0]['params'])}")  # 0

Number of parameters: 0
Requires grad on weight? False


ValueError: optimizer got an empty parameter list

In [ ]:
class Linear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        # These are automatically registered as parameters
        self.weight = nn.Parameter(torch.Tensor(out_features, in_features))
        self.bias = nn.Parameter(torch.Tensor(out_features))
        
        # Alternative manual registration:
        # self.register_parameter('weight', nn.Parameter(...))

## Module Hierarchy: Parameters

In [ ]:
class Network(nn.Module):
    def __init__(self):
        super().__init__()
        # Child modules are automatically registered
        self.layer1 = nn.Linear(10, 20)
        self.layer2 = nn.Linear(20, 5)
        
    def forward(self, x):
        return self.layer2(self.layer1(x))
        
# Accessing the hierarchy:
model = Network()
print(list(model.children()))  # [layer1, layer2]


In [ ]:
print(list(model.parameters()))  # All parameters from both layers

In [ ]:
class CustomLayer(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_features, out_features))
        self.bias = nn.Parameter(torch.zeros(out_features))
    
    def forward(self, x):
        return x @ self.weight + self.bias

## An Example of a Complex Network

In [ ]:
class ComplexNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Submodule 1: Feature extractor
        self.feature_extractor = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # Submodule 2: Processor
        self.processor = nn.ModuleList([
            nn.Linear(256, 128),
            nn.Linear(128, 64)
        ])
        
        # Submodule 3: Output head
        self.classifier = nn.Linear(64, 10)
        
    def forward(self, x):
        x = self.feature_extractor(x)
        for layer in self.processor:
            x = layer(x)
        return self.classifier(x)

# Inspecting the hierarchy
model = ComplexNetwork()
print(f"Number of parameters: {sum(p.numel() for p in model.parameters())}")
print(f"Module structure:\n{model}")

# Accessing specific parts
print(f"Feature extractor parameters: {list(model.feature_extractor.parameters())}")

## A Layer with a Conditional Parameter

In [ ]:
class ConditionalLayer(nn.Module):
    """Layer that sometimes has bias, sometimes doesn't"""
    def __init__(self, in_dim, out_dim, use_bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_dim, in_dim))
        self.use_bias = use_bias
        
        if use_bias:
            self.bias = nn.Parameter(torch.zeros(out_dim))
        else:
            # Important: Register bias as None
            self.register_parameter('bias', None)
            
    def forward(self, x):
        result = x @ self.weight.T
        if self.bias is not None:
            result = result + self.bias
        return result

# Both versions work correctly with optimizers
layer1 = ConditionalLayer(10, 5, use_bias=True)
layer2 = ConditionalLayer(10, 5, use_bias=False)
print(f"Layer1 params: {len(list(layer1.parameters()))}")  # 2
print(f"Layer2 params: {len(list(layer2.parameters()))}")  # 1

## Hooks

### Forward Hook

In [6]:
import torch
import torch.nn as nn

# Define a simple model
model = nn.Linear(5, 3)

# Define a hook function
def print_output(module, input, output):
    print(module)
    print(f"Iput: {input}")
    print(f"Output: {output}")

# Register the hook
handle = model.register_forward_hook(print_output)

# Run the model
x = torch.randn(1, 5)
print(x)
output = model(x)

# Remove the hook
handle.remove()

tensor([[ 1.47666156,  0.84247339, -1.61149466,  0.28967109, -0.62255901]])
Linear(in_features=5, out_features=3, bias=True)
Iput: (tensor([[ 1.47666156,  0.84247339, -1.61149466,  0.28967109, -0.62255901]]),)
Output: tensor([[ 0.04884279,  0.13909318, -0.44066066]], grad_fn=<AddmmBackward0>)


### Backward Hook

In [12]:
import torch
import torch.nn as nn

model = nn.Linear(5, 3)
model.weight.data = torch.tensor([
    [0.1, 0.2, 0.3, 0.4, 0.5],
    [0.6, 0.7, 0.8, 0.9, 1.0],
    [1.1, 1.2, 1.3, 1.4, 1.5]
])
model.bias.data = torch.tensor([0.1, 0.2, 0.3])

def print_gradients(module, grad_input, grad_output):
    print(f"Grad Input: {grad_input}")
    print(f"Grad Output: {grad_output}")

handle = model.register_backward_hook(print_gradients)

x = torch.tensor([[1., 2., 3., 4., 5.]])
print(x)
output = model(x)
loss = output.sum()
loss.backward()

handle.remove()

tensor([[1., 2., 3., 4., 5.]])
Grad Input: (tensor([1., 1., 1.]), None, tensor([[1., 1., 1.],
        [2., 2., 2.],
        [3., 3., 3.],
        [4., 4., 4.],
        [5., 5., 5.]]))
Grad Output: (tensor([[1., 1., 1.]]),)


Grad Input Tuple: (1): initial gradient input (2) Gradient with respect to bias (not provided by Hook), (3) Initial Gradient with respect to W (4) Out gradient with respect to W.

In [13]:
print("Weight gradient:", model.weight.grad)
print("Bias gradient:", model.bias.grad)

Weight gradient: tensor([[1., 2., 3., 4., 5.],
        [1., 2., 3., 4., 5.],
        [1., 2., 3., 4., 5.]])
Bias gradient: tensor([1., 1., 1.])


In [16]:
import torch
import torch.nn as nn

class HookedModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 5)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(5, 2)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# 1. Define the Forward Hook (peeking at activations)
def forward_hook_fn(module, input, output):
    print(f"--- Forward Hook: {module.__class__.__name__} ---")
    print(f"Input Shape: {input[0].shape}")
    print(f"Output Shape: {output.shape}\n")

# 2. Define the Backward Hook (peeking at gradients)
def backward_hook_fn(module, grad_input, grad_output):
    print(f"--- Backward Hook: {module.__class__.__name__} ---")
    # grad_output is the gradient of the loss w.r.t the output of this layer
    print(f"Gradient Output Norm: {grad_output[0].norm().item():.4f}\n")

# --- Execution ---
model = HookedModel()

# Register hooks on the ReLU layer specifically
handle_f = model.relu.register_forward_hook(forward_hook_fn)
handle_b = model.relu.register_full_backward_hook(backward_hook_fn)

# Dummy Input
data = torch.randn(1, 10)
target = torch.randn(1, 2)

# Forward Pass
output = model(data)

# Backward Pass
loss = torch.nn.functional.mse_loss(output, target)
loss.backward()

# Clean up (it's good practice to remove hooks after debugging)
handle_f.remove()
handle_b.remove()

--- Forward Hook: ReLU ---
Input Shape: torch.Size([1, 5])
Output Shape: torch.Size([1, 5])

--- Backward Hook: ReLU ---
Gradient Output Norm: 1.5828



In [ ]:
# df = pl.read_csv("https://raw.githubusercontent.com/rahulbhadani/CPE487587_SP26/refs/heads/master/Data/ResourceAssessmentSummaryData032011.csv",
#                 schema_overrides={
#         "Design Head (feet) ": pl.Utf8,
#         "Design Flow (cfs)": pl.Utf8,
#         "Installed Capacity (kW)": pl.Utf8,
#         "Annual Production (MWh)": pl.Utf8,
#         "Plant Factor": pl.Utf8,
#         "Total Construction Cost (1,000 $)": pl.Utf8,
#         "Annual O&M Cost (1,000 $)": pl.Utf8,
#         "Cost per Installed Capacity ($/kW)": pl.Utf8,
#         "IRR with Green Incentives": pl.Utf8,
#     }
# )

# # Remove quotes, commas, and dollar signs, then convert to float
# df = df.with_columns(
#     pl.col([
#         "Design Head (feet) ",
#         "Design Flow (cfs)",
#         "Installed Capacity (kW)",
#         "Annual Production (MWh)",
#         "Plant Factor",
#         "Total Construction Cost (1,000 $)",
#         "Annual O&M Cost (1,000 $)",
#         "Cost per Installed Capacity ($/kW)",
#         "IRR with Green Incentives",
#     ])
#     .str.replace_all(r'["\$,]', '')  # remove quotes, $, and commas
#     .str.replace_all(r'[<>]', '')     # remove < and >
#     .cast(pl.Float64)
# )

In [ ]:
# df = pl.read_csv("https://raw.githubusercontent.com/rahulbhadani/CPE487587_SP26/refs/heads/master/Data/ResourceAssessmentSummaryData032011.csv",
#                 schema_overrides={
#         "Design Head (feet) ": pl.Utf8,
#         "Design Flow (cfs)": pl.Utf8,
#         "Installed Capacity (kW)": pl.Utf8,
#         "Annual Production (MWh)": pl.Utf8,
#         "Plant Factor": pl.Utf8,
#         "Total Construction Cost (1,000 $)": pl.Utf8,
#         "Annual O&M Cost (1,000 $)": pl.Utf8,
#         "Cost per Installed Capacity ($/kW)": pl.Utf8,
#         "IRR with Green Incentives": pl.Utf8,
#     }
# )

# # Remove quotes, commas, and dollar signs, then convert to float
# df = df.with_columns(
#     pl.col([
#         "Design Head (feet) ",
#         "Design Flow (cfs)",
#         "Installed Capacity (kW)",
#         "Annual Production (MWh)",
#         "Plant Factor",
#         "Total Construction Cost (1,000 $)",
#         "Annual O&M Cost (1,000 $)",
#         "Cost per Installed Capacity ($/kW)",
#         "IRR with Green Incentives",
#     ])
#     .str.replace_all(r'["\$,]', '')  # remove quotes, $, and commas
#     .str.replace_all(r'[<>]', '')     # remove < and >
#     .cast(pl.Float64)
# )

# 2. Neural Network Training

In [2]:
df = pl.read_csv("https://raw.githubusercontent.com/rahulbhadani/CPE486586_FA25/refs/heads/main/Data/Concrete_Compressive_Strength/Concrete_Data.csv")

df

Cement (component 1)(kg in a m^3 mixture),Blast Furnace Slag (component 2)(kg in a m^3 mixture),Fly Ash (component 3)(kg in a m^3 mixture),Water (component 4)(kg in a m^3 mixture),Superplasticizer (component 5)(kg in a m^3 mixture),Coarse Aggregate (component 6)(kg in a m^3 mixture),Fine Aggregate (component 7)(kg in a m^3 mixture),Age (day),"Concrete compressive strength(MPa, megapascals)"
str,str,str,str,str,str,str,str,str
"""540.0 ""","""0.0 ""","""0.0 ""","""162.0 ""","""2.5 ""","""1040.0 ""","""676.0 ""","""28 ""","""79.99 """
"""540.0 ""","""0.0 ""","""0.0 ""","""162.0 ""","""2.5 ""","""1055.0 ""","""676.0 ""","""28 ""","""61.89 """
"""332.5 ""","""142.5 ""","""0.0 ""","""228.0 ""","""0.0 ""","""932.0 ""","""594.0 ""","""270 ""","""40.27 """
"""332.5 ""","""142.5 ""","""0.0 ""","""228.0 ""","""0.0 ""","""932.0 ""","""594.0 ""","""365 ""","""41.05 """
"""198.6 ""","""132.4 ""","""0.0 ""","""192.0 ""","""0.0 ""","""978.4 ""","""825.5 ""","""360 ""","""44.30 """
…,…,…,…,…,…,…,…,…
"""276.4 ""","""116.0 ""","""90.3 ""","""179.6 ""","""8.9 ""","""870.1 ""","""768.3 ""","""28 ""","""44.28 """
"""322.2 ""","""0.0 ""","""115.6 ""","""196.0 ""","""10.4 ""","""817.9 ""","""813.4 ""","""28 ""","""31.18 """
"""148.5 ""","""139.4 ""","""108.6 ""","""192.7 ""","""6.1 ""","""892.4 ""","""780.0 ""","""28 ""","""23.70 """


In [3]:
# Rename
df.columns = ['Cement', 'BlastFuranceSlag', 'FlyAsh', 'Water', 'Superplasticizer', 'CoarseAggregate', 'FineAggregate', 'Age',  'ConcreteStrength']

In [4]:
df

Cement,BlastFuranceSlag,FlyAsh,Water,Superplasticizer,CoarseAggregate,FineAggregate,Age,ConcreteStrength
str,str,str,str,str,str,str,str,str
"""540.0 ""","""0.0 ""","""0.0 ""","""162.0 ""","""2.5 ""","""1040.0 ""","""676.0 ""","""28 ""","""79.99 """
"""540.0 ""","""0.0 ""","""0.0 ""","""162.0 ""","""2.5 ""","""1055.0 ""","""676.0 ""","""28 ""","""61.89 """
"""332.5 ""","""142.5 ""","""0.0 ""","""228.0 ""","""0.0 ""","""932.0 ""","""594.0 ""","""270 ""","""40.27 """
"""332.5 ""","""142.5 ""","""0.0 ""","""228.0 ""","""0.0 ""","""932.0 ""","""594.0 ""","""365 ""","""41.05 """
"""198.6 ""","""132.4 ""","""0.0 ""","""192.0 ""","""0.0 ""","""978.4 ""","""825.5 ""","""360 ""","""44.30 """
…,…,…,…,…,…,…,…,…
"""276.4 ""","""116.0 ""","""90.3 ""","""179.6 ""","""8.9 ""","""870.1 ""","""768.3 ""","""28 ""","""44.28 """
"""322.2 ""","""0.0 ""","""115.6 ""","""196.0 ""","""10.4 ""","""817.9 ""","""813.4 ""","""28 ""","""31.18 """
"""148.5 ""","""139.4 ""","""108.6 ""","""192.7 ""","""6.1 ""","""892.4 ""","""780.0 ""","""28 ""","""23.70 """


Clearly, Polars didn't do a good job of reading dataset properly, so we need to manually strip out white spaces, and read as float.

In [5]:
# 3. Clean and Cast all columns to Float32
df = df.with_columns(
    pl.all().str.strip_chars().cast(pl.Float32)
)
df

Cement,BlastFuranceSlag,FlyAsh,Water,Superplasticizer,CoarseAggregate,FineAggregate,Age,ConcreteStrength
f32,f32,f32,f32,f32,f32,f32,f32,f32
540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28.0,79.989998
540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28.0,61.889999
332.5,142.5,0.0,228.0,0.0,932.0,594.0,270.0,40.27
332.5,142.5,0.0,228.0,0.0,932.0,594.0,365.0,41.049999
198.600006,132.399994,0.0,192.0,0.0,978.400024,825.5,360.0,44.299999
…,…,…,…,…,…,…,…,…
276.399994,116.0,90.300003,179.600006,8.9,870.099976,768.299988,28.0,44.279999
322.200012,0.0,115.599998,196.0,10.4,817.900024,813.400024,28.0,31.18
148.5,139.399994,108.599998,192.699997,6.1,892.400024,780.0,28.0,23.700001


Casting Strings to Float can be problematic, but we are gonna live with it for the time being

In [6]:
import polars as pl

# Float32 has limited precision
value_f32 = pl.Series([79.99]).cast(pl.Float32)[0]
print(value_f32)  # 79.989998 (precision lost!)

79.98999786376953


We want to estimat annual production based BCR, ConstructionCost and DesignHead

In [7]:
class SimpleNN(nn.Module):
    def __init__(self, in_features):
        super(SimpleNN, self).__init__()
        self.in_features = in_features
        self.fc1 = nn.Linear(self.in_features, 64)
        self.fc2 = nn.Linear(64, 128)
        self.fc3 = nn.Linear(128, 16)
        self.fc4 = nn.Linear(16, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)
        return x

In [8]:
# Convert Polars DataFrame to numpy arrays
X = df.drop(['ConcreteStrength']).to_numpy() 
y = df['ConcreteStrength'].to_numpy()     

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [21]:
y.shape

(1030,)

In [17]:
y_train.reshape(-1,1)

array([[27.68],
       [62.05],
       [23.8 ],
       [33.4 ],
       [ 7.4 ],
       [27.77],
       [18.29],
       [48.59],
       [39.7 ],
       [ 4.57],
       [13.29],
       [36.97],
       [22.53],
       [71.3 ],
       [25.61],
       [76.24],
       [62.94],
       [17.54],
       [41.05],
       [21.86],
       [47.13],
       [16.5 ],
       [22.72],
       [29.72],
       [19.93],
       [ 9.62],
       [39.05],
       [42.13],
       [39.32],
       [34.49],
       [28.1 ],
       [38.6 ],
       [53.77],
       [ 7.32],
       [32.82],
       [43.38],
       [55.16],
       [35.23],
       [66.7 ],
       [30.88],
       [76.8 ],
       [17.96],
       [55.06],
       [64.3 ],
       [33.8 ],
       [45.94],
       [37.26],
       [24.85],
       [40.15],
       [13.54],
       [32.88],
       [17.57],
       [21.54],
       [17.84],
       [23.4 ],
       [55.55],
       [17.6 ],
       [31.42],
       [13.82],
       [65.91],
       [81.75],
       [28.8 ],
       [

In [18]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32, device=device)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32, device=device)
y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32, device=device)
y_test_tensor = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32, device=device)

In [19]:
model = SimpleNN(in_features = X_train_tensor.shape[1]).to(device)


In [20]:
# Define the loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.00001)
# Train the model
epochs = 100000

losses = torch.zeros(epochs, device=device)

for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    
    losses[epoch] = loss
    optimizer.step()
    if (epoch + 1) % 1000 == 0:
        print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item()}')
        print(f'GPU Memory: {torch.cuda.memory_allocated() / 1024**2:.5f} MB')

Epoch 1000/100000, Loss: 1524.3343505859375
GPU Memory: 17.84863 MB
Epoch 2000/100000, Loss: 1365.3143310546875
GPU Memory: 17.84863 MB
Epoch 3000/100000, Loss: 1070.5272216796875
GPU Memory: 17.84863 MB
Epoch 4000/100000, Loss: 713.7730102539062
GPU Memory: 17.84863 MB
Epoch 5000/100000, Loss: 418.3660583496094
GPU Memory: 17.84863 MB
Epoch 6000/100000, Loss: 272.9963684082031
GPU Memory: 17.84863 MB
Epoch 7000/100000, Loss: 233.36376953125
GPU Memory: 17.84863 MB
Epoch 8000/100000, Loss: 218.1627197265625
GPU Memory: 17.84863 MB
Epoch 9000/100000, Loss: 205.78553771972656
GPU Memory: 17.84863 MB
Epoch 10000/100000, Loss: 193.96908569335938
GPU Memory: 17.84863 MB
Epoch 11000/100000, Loss: 182.45156860351562
GPU Memory: 17.84863 MB
Epoch 12000/100000, Loss: 171.2572021484375
GPU Memory: 17.84863 MB
Epoch 13000/100000, Loss: 160.407470703125
GPU Memory: 17.84863 MB
Epoch 14000/100000, Loss: 149.72840881347656
GPU Memory: 17.84863 MB
Epoch 15000/100000, Loss: 138.6429443359375
GPU Memor

### Testing

In [26]:
torch.__version__

'2.9.1+cu128'

In [27]:
!uv add torchmetrics

Resolved 169 packages in 714ms                                       
⠙ Preparing packages... (0/2)                                                   
⠙ Preparing packages... (0/2)------     0 B/960.12 KiB                  
⠙ Preparing packages... (0/2)------ 14.91 KiB/960.12 KiB                
⠙ Preparing packages... (0/2)------ 30.91 KiB/960.12 KiB                
⠙ Preparing packages... (0/2)------ 46.91 KiB/960.12 KiB                
⠙ Preparing packages... (0/2)------ 62.91 KiB/960.12 KiB                
⠙ Preparing packages... (0/2)------ 78.85 KiB/960.12 KiB                
⠙ Preparing packages... (0/2)------ 94.85 KiB/960.12 KiB                
lightning-utilities ------------------------------     0 B/28.74 KiB
⠙ Preparing packages... (0/2)------ 94.85 KiB/960.12 KiB                
lightning-utilities ------------------------------     0 B/28.74 KiB
⠙ Preparing packages... (0/2)------ 110.85 KiB/960.12 KiB               
lightning-utilities ------------------------------    

In [30]:
# set to model.eval()
model.eval()

with torch.no_grad():
    y_pred = model(X_test_tensor)
        
loss = criterion(y_pred, y_test_tensor)
print(loss)

from torchmetrics.functional.regression import r2_score

r2 = r2_score(y_pred, y_test_tensor)
print(f'R-squared on the test data: {r2.item()}')

with torch.no_grad():
    y_pred_train = model(X_train_tensor)
        

training_loss = criterion(y_pred_train, y_train_tensor)
print(training_loss)

r2 = r2_score(y_pred_train, y_train_tensor)
print(f'R-squared on the train data: {r2.item()}')

tensor(37.93627167, device='cuda:0')
R-squared on the test data: 0.8527757525444031
tensor(1.46585000, device='cuda:0')
R-squared on the train data: 0.994840145111084


## Batch Training

In [31]:
from torch.utils.data import TensorDataset, DataLoader

In [32]:
# Create TensorDataset and DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Set batch size
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [33]:
# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleNN(in_features=X_train_tensor.shape[1]).to(device)

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.00001) 

# Training with batching
epochs = 100000
train_losses = []
val_losses = []

for epoch in range(epochs):
    # Training phase
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Validation phase
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(test_loader)
    val_losses.append(avg_val_loss)
    
    if (epoch + 1) % 1000 == 0:
        print(f'Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}')
        if torch.cuda.is_available():
            print(f'GPU Memory: {torch.cuda.memory_allocated() / 1024**2:.2f} MB')

# Evaluate on full test set
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor.to(device))
    test_loss = criterion(test_outputs, y_test_tensor.to(device))
    print(f'\nFinal Test Loss: {test_loss.item():.6f}')

Epoch 1000/100000, Train Loss: 131.418535, Val Loss: 123.958711
GPU Memory: 18.10 MB
Epoch 2000/100000, Train Loss: 64.994687, Val Loss: 67.753150
GPU Memory: 18.10 MB
Epoch 3000/100000, Train Loss: 43.286705, Val Loss: 47.492290
GPU Memory: 18.10 MB
Epoch 4000/100000, Train Loss: 34.797856, Val Loss: 41.524952
GPU Memory: 18.10 MB
Epoch 5000/100000, Train Loss: 29.227293, Val Loss: 38.360862
GPU Memory: 18.10 MB
Epoch 6000/100000, Train Loss: 25.247484, Val Loss: 36.815020
GPU Memory: 18.10 MB
Epoch 7000/100000, Train Loss: 21.849866, Val Loss: 35.518548
GPU Memory: 18.10 MB
Epoch 8000/100000, Train Loss: 19.348029, Val Loss: 34.690653
GPU Memory: 18.10 MB
Epoch 9000/100000, Train Loss: 17.365025, Val Loss: 33.871991
GPU Memory: 18.10 MB
Epoch 10000/100000, Train Loss: 15.569677, Val Loss: 33.169832
GPU Memory: 18.10 MB
Epoch 11000/100000, Train Loss: 14.038964, Val Loss: 32.930327
GPU Memory: 18.10 MB
Epoch 12000/100000, Train Loss: 13.099302, Val Loss: 32.851543
GPU Memory: 18.10 MB

Evaluation

In [34]:
# set to model.eval()
model.eval()

with torch.no_grad():
    y_pred = model(X_test_tensor)
        
loss = criterion(y_pred, y_test_tensor)
print(loss)

from torchmetrics.functional.regression import r2_score

r2 = r2_score(y_pred, y_test_tensor)
print(f'R-squared on the test data: {r2.item()}')

with torch.no_grad():
    y_pred_train = model(X_train_tensor)
        

training_loss = criterion(y_pred_train, y_train_tensor)
print(training_loss)

r2 = r2_score(y_pred_train, y_train_tensor)
print(f'R-squared on the train data: {r2.item()}')

tensor(26.13053894, device='cuda:0')
R-squared on the test data: 0.8985918164253235
tensor(1.80286086, device='cuda:0')
R-squared on the train data: 0.9936538338661194


## Batch Normalization

In [37]:
class SimpleNNWithBN(nn.Module):
    def __init__(self, in_features):
        super(SimpleNNWithBN, self).__init__()
        self.in_features = in_features
        
        # Layers with Batch Normalization
        self.fc1 = nn.Linear(self.in_features, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.fc2 = nn.Linear(64, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.fc3 = nn.Linear(128, 16)
        self.bn3 = nn.BatchNorm1d(16)
        self.fc4 = nn.Linear(16, 1)  # Output layer - no BN
        
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2) 

    def forward(self, x):
        # Layer 1: Linear -> BatchNorm -> Activation
        x = self.fc1(x)
        x = self.bn1(x)  # BatchNorm before activation
        x = self.relu(x)
        x = self.dropout(x) 
        
        # Layer 2
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.dropout(x)
        
        # Layer 3
        x = self.fc3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.dropout(x)
        
        # Output layer (no BN, no activation for regression)
        x = self.fc4(x)
        return x

In [38]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Set batch size
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [42]:
model = SimpleNNWithBN(in_features=X_train_tensor.shape[1]).to(device)

# Training parameters
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)  # Added weight decay
epochs = 20000


In [43]:
rain_losses = []
val_losses = []

for epoch in range(epochs):
    # Training phase
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    # Validation phase
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(test_loader)
    val_losses.append(avg_val_loss)
    
    if (epoch + 1) % 1000 == 0:
        print(f'Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}')
        if torch.cuda.is_available():
            print(f'GPU Memory: {torch.cuda.memory_allocated() / 1024**2:.2f} MB')

# Evaluate on full test set
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor.to(device))
    test_loss = criterion(test_outputs, y_test_tensor.to(device))
    print(f'\nFinal Test Loss: {test_loss.item():.6f}')

Epoch 1000/20000, Train Loss: 53.945469, Val Loss: 32.691339
GPU Memory: 17.96 MB
Epoch 2000/20000, Train Loss: 43.270767, Val Loss: 29.541453
GPU Memory: 17.96 MB
Epoch 3000/20000, Train Loss: 42.219031, Val Loss: 39.867542
GPU Memory: 17.96 MB
Epoch 4000/20000, Train Loss: 38.762968, Val Loss: 36.955585
GPU Memory: 17.96 MB
Epoch 5000/20000, Train Loss: 39.952301, Val Loss: 43.152736
GPU Memory: 17.96 MB
Epoch 6000/20000, Train Loss: 33.930105, Val Loss: 36.294400
GPU Memory: 17.96 MB
Epoch 7000/20000, Train Loss: 39.613232, Val Loss: 46.438476
GPU Memory: 17.96 MB
Epoch 8000/20000, Train Loss: 35.198731, Val Loss: 36.757748
GPU Memory: 17.96 MB
Epoch 9000/20000, Train Loss: 42.235152, Val Loss: 36.003036
GPU Memory: 17.96 MB
Epoch 10000/20000, Train Loss: 38.409001, Val Loss: 37.901125
GPU Memory: 17.96 MB
Epoch 11000/20000, Train Loss: 41.054450, Val Loss: 44.013346
GPU Memory: 17.96 MB
Epoch 12000/20000, Train Loss: 37.386356, Val Loss: 43.335606
GPU Memory: 17.96 MB
Epoch 13000/2